# Load & Parse Data

In [2]:
import importlib
import utils.parse_data  

importlib.reload(utils.parse_data)
from utils.parse_data import load_dataset_main

In [3]:
from utils.parse_data import load_dataset_main

dataset, device = load_dataset_main()

Using dataset path: data\pooled_stratified_share
Loading dataset with fixed schema settings...
Scanning and loading dataset (optimized fixed schema mode)...
Loaded: 2144 runs from 6 subjects
Subjects: ['subjOenz4SHyaO', 'subjQX6R7rImeb', 'subjoVC0sHUB_P', 'subjqFdsBbuiHF', 'subjtySNgRhV65', 'subjxpYwO4azeZ']
Successfully loaded runs: 2144
Loading time: 28.80 seconds
Device: cuda


# Train Model

In [5]:
import importlib
import utils.training

importlib.reload(utils.training)

<module 'utils.training' from 'C:\\Users\\bayer\\PycharmProjects\\fmri_connectivity\\forecasting\\utils\\training.py'>

In [6]:
from utils.training import run_loso_cv
from models.lstm.lstm_model_library import alstm_model_generator
from models.linear_regression.linear_regression_core import linear_regression_generator

n_roi, H = 19, 3
alpha = 1.0

model_gen = lambda : alstm_model_generator(n_roi, H)
model_gen = lambda : linear_regression_generator(alpha)


results_df, model, X_test, Y_test, \
best_model, best_X_test, best_Y_test = run_loso_cv(
    dataset_raw=dataset,
    model_gen=model_gen,
    M=50,
    H=3,
    stride=1,
    num_epochs=20,
    batch_size=512,
    device=device
)


LOSO-CV | 6 subjects | M=50, H=3


LOSO Folds:   0%|          | 0/6 [00:00<?, ?it/s]


Fold 1/6 | Test subject: subjOenz4SHyaO
Train subjects: ['subjQX6R7rImeb', 'subjoVC0sHUB_P', 'subjqFdsBbuiHF', 'subjtySNgRhV65', 'subjxpYwO4azeZ']
Test subjects : ['subjOenz4SHyaO']
Train runs: 1744
Test runs : 400
Train windows: 273110 | Val windows: 30346 | Test windows: 69600 | ROIs: 19
Model is not CUDA compatib: 'Ridge' object has no attribute 'to'
Conintuing with CPU...

Fold 2/6 | Test subject: subjQX6R7rImeb
Train subjects: ['subjOenz4SHyaO', 'subjoVC0sHUB_P', 'subjqFdsBbuiHF', 'subjtySNgRhV65', 'subjxpYwO4azeZ']
Test subjects : ['subjQX6R7rImeb']
Train runs: 1840
Test runs : 304
Train windows: 288144 | Val windows: 32016 | Test windows: 52896 | ROIs: 19
Model is not CUDA compatib: 'Ridge' object has no attribute 'to'
Conintuing with CPU...

Fold 3/6 | Test subject: subjoVC0sHUB_P
Train subjects: ['subjOenz4SHyaO', 'subjQX6R7rImeb', 'subjqFdsBbuiHF', 'subjtySNgRhV65', 'subjxpYwO4azeZ']
Test subjects : ['subjoVC0sHUB_P']
Train runs: 1840
Test runs : 304
Train windows: 288144 | 

KeyError: 'LSTM_RMSE'

In [7]:
best_model

NameError: name 'best_model' is not defined

In [3]:
dataset

[{'timeseries': array([[ 55.090908 , 182.66667  , 466.       , ...,   1.9310344,
          673.8148   , 562.97144  ],
         [ 53.636364 , 178.16667  , 458.5      , ...,   1.862069 ,
          671.8889   , 564.4286   ],
         [ 52.909092 , 184.33333  , 450.5      , ...,   1.8965517,
          668.2222   , 565.0571   ],
         ...,
         [ 49.81818  , 159.5      , 414.5      , ...,   1.7931035,
          675.14813  , 550.3714   ],
         [ 49.545456 , 157.       , 415.5      , ...,   1.8275862,
          675.14813  , 549.3714   ],
         [ 48.909092 , 158.       , 425.5      , ...,   1.8965517,
          673.14813  , 556.54285  ]], shape=(226, 19), dtype=float32),
  'subject': 'subjOenz4SHyaO',
  'roi_labels': ('1',
   '2',
   '3',
   '4',
   '5',
   '6',
   '7',
   '8',
   '9',
   '10',
   '11',
   '12',
   '13',
   '14',
   '15',
   '16',
   '17',
   '18',
   '19')},
 {'timeseries': array([[ 48.363636 , 185.16667  , 404.5      , ...,   1.8275862,
          668.1111   , 5

# Best Model

In [6]:
import importlib
import utils.parse_data

importlib.reload(utils.parse_data)

<module 'utils.parse_data' from 'C:\\Users\\bayer\\PycharmProjects\\fmri_connectivity\\forecasting\\utils\\parse_data.py'>

In [7]:
from utils.parse_data import parse_dataset

X_train, Y_train, X_test, Y_test, device = parse_dataset(M=50, H=3, normalize=True, stride=1, test_ratio=0.2, 
                                                         test_subjects=None, random_state=42, verbose=True)

Using dataset path: data\pooled_stratified_share
Loading dataset with fixed schema settings...
Scanning and loading dataset (optimized fixed schema mode)...
Loaded: 2144 runs from 6 subjects
Subjects: ['subjOenz4SHyaO', 'subjQX6R7rImeb', 'subjoVC0sHUB_P', 'subjqFdsBbuiHF', 'subjtySNgRhV65', 'subjxpYwO4azeZ']
Successfully loaded runs: 2144
Loading time: 3.30 seconds
Device: cuda
Normalizing dataset...
Building sliding windows...
Splitting by subject...
Train subjects: ['subjqFdsBbuiHF', 'subjoVC0sHUB_P', 'subjxpYwO4azeZ', 'subjtySNgRhV65']
Test subjects : ['subjQX6R7rImeb', 'subjOenz4SHyaO']
Train runs: 1440
Test runs : 704
Final shapes - Train: (250560, 50, 19), (250560, 3, 19) | Test: (122496, 50, 19), (122496, 3, 19)


In [14]:
import joblib
import numpy as np
import torch
from models.lstm.model_library import AdvancedLSTM, FmriPredictorAPI
from pathlib import Path

#device = torch.device('cpu')

# 1. Load the packaged model
model = joblib.load("saves/best_lstm.joblib")
#model = torch.load('best_fmri_sklearn_api.joblib', map_location=device)

# 2. Prepare your data (Example: 1 window of 50 seconds for 19 ROIs)
# Ensure your real data is Z-score normalized!
#my_data = np.random.randn(1, 50, 19) 

# 3. Predict
# Output: Forecast for the next 5 seconds (H=5) -> Shape: (1, 5, 19)
predictions = model.predict(X_test)

print("Inference completed successfully.")
print(f"Forecast for the next {H} steps: {predictions}")

ModuleNotFoundError: No module named 'LSTM_model_library'

In [2]:
import torch
print(torch.cuda.is_available())


True


In [5]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.version.cuda)

2.11.0+cu130
False
0
13.0
